In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# ייצוג מחשבים קוונטיים עבור ה-Transpiler


<details>
<summary><b>Package versions</b></summary>

The code on this page was developed using the following requirements.
We recommend using these versions or newer.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
כדי להמיר Circuit מופשט ל-Circuit ISA שיכול לרוץ על QPU ספציפי (יחידת עיבוד קוונטי), ה-Transpiler זקוק למידע מסוים על ה-QPU. מידע זה נמצא בשני מקומות: אובייקט `BackendV2` (או `BackendV1` הישן) שאתה מתכנן להגיש אליו עבודות, ותכונת ה-`Target` של ה-Backend.

- ה-[`Target`](../api/qiskit/qiskit.transpiler.Target) מכיל את כל האילוצים הרלוונטיים של מכשיר, כגון Gate הבסיסיים הטבעיים שלו, קישוריות Qubit, ומידע פולס או תזמון.
- ה-[`Backend`](../api/qiskit/qiskit.providers.BackendV2) מחזיק `Target` כברירת מחדל, מכיל מידע נוסף -- כגון [`InstructionScheduleMap`](https://docs.quantum.ibm.com/api/qiskit/1.4/qiskit.pulse.InstructionScheduleMap), ומספק את הממשק להגשת עבודות Circuit קוונטי.

תוכל גם לספק מידע במפורש עבור ה-Transpiler לשימוש, לדוגמה, אם יש לך מקרה שימוש ספציפי, או אם אתה מאמין שמידע זה יסייע ל-Transpiler לייצר Circuit מאופטם יותר.

הדיוק שבו ה-Transpiler מייצר את ה-Circuit המתאים ביותר לחומרה ספציפית תלוי בכמה מידע יש ל-`Target` או ל-`Backend` על האילוצים שלהם.

> **Note:** מכיוון שאלגוריתמים רבים מהאלגוריתמים הבסיסיים של הקומפילציה הם סטוכסטיים, אין ערובה שיימצא Circuit טוב יותר.

דף זה מציג מספר דוגמאות של העברת מידע QPU ל-Transpiler. דוגמאות אלו משתמשות ב-target מה-Backend המדומה [`FakeSherbrooke`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/fake-provider-fake-sherbrooke#fakesherbrooke).

<span id="default-config"></span>
## תצורת ברירת מחדל
השימוש הפשוט ביותר ב-Transpiler הוא לספק את כל מידע ה-QPU על ידי מתן ה-`Backend` או `Target`. כדי להבין טוב יותר כיצד ה-Transpiler עובד, בנה Circuit ועבד אותו עם מידע שונה, כדלקמן.

ייבא את הספריות הנדרשות ואתחל את ה-QPU:
כדי להמיר Circuit מופשט ל-Circuit ISA שיכול לרוץ על מעבד ספציפי, ה-Transpiler זקוק למידע מסוים על המעבד. בדרך כלל, מידע זה מאוחסן ב-[`Backend`](https://docs.quantum.ibm.com/api/qiskit/qiskit.providers.Backend#backend) או ב-[`Target`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.Target#target) שסופק ל-Transpiler, ואין צורך במידע נוסף. עם זאת, תוכל גם לספק מידע במפורש עבור ה-Transpiler לשימוש, לדוגמה, אם יש לך מקרה שימוש ספציפי, או אם אתה מאמין שמידע זה יסייע ל-Transpiler לייצר Circuit מאופטם יותר.

נושא זה מציג מספר דוגמאות של העברת מידע ל-Transpiler. דוגמאות אלו משתמשות ב-target מה-Backend המדומה [`FakeSherbrooke`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/fake-provider-fake-sherbrooke#fakesherbrooke).

In [1]:
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

backend = FakeSherbrooke()
target = backend.target

ה-Circuit לדוגמה משתמש במופע של [`efficient_su2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.efficient_su2) מספריית ה-Circuit של Qiskit.

In [2]:
from qiskit.circuit.library import efficient_su2

qc = efficient_su2(12, entanglement="circular", reps=1)

qc.draw("mpl")

<Image src="../docs/images/guides/represent-quantum-computers/extracted-outputs/97f9acc1-ac53-4025-b413-485777932a9b-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/represent-quantum-computers/extracted-outputs/97f9acc1-ac53-4025-b413-485777932a9b-0.svg)

דוגמה זו משתמשת בהגדרות ברירת מחדל לעיבוד ל-`target` של ה-`backend`, המספקת את כל המידע הדרוש להמרת ה-Circuit לכזה שיפעל על ה-Backend.

In [3]:
from qiskit.transpiler import generate_preset_pass_manager

pass_manager = generate_preset_pass_manager(
    optimization_level=1, target=target, seed_transpiler=12345
)
qc_t_target = pass_manager.run(qc)
qc_t_target.draw("mpl", idle_wires=False, fold=-1)

<Image src="../docs/images/guides/represent-quantum-computers/extracted-outputs/4b81fb9d-d199-45c5-b119-c1f0b973afe9-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/represent-quantum-computers/extracted-outputs/4b81fb9d-d199-45c5-b119-c1f0b973afe9-0.svg)

דוגמה זו משמשת בסעיפים מאוחרים יותר של נושא זה להמחשה שמפת הקישוריות ו-Gate הבסיס הם מקטעי המידע החיוניים להעביר ל-Transpiler לבנייה אופטימלית של Circuit. ה-QPU יכול בדרך כלל לבחור הגדרות ברירת מחדל למידע אחר שאינו מועבר, כגון תזמון ולוח זמנים.

## מפת קישוריות

מפת הקישוריות היא גרף המראה אילו Qubit מחוברים ולכן יש ביניהם Gate דו-Qubit. לפעמים גרף זה הוא כיווני, כלומר Gate דו-Qubit יכולים לנוע רק בכיוון אחד. עם זאת, ה-Transpiler תמיד יכול להפוך את כיוון ה-Gate על ידי הוספת Gate חד-Qubit נוספים. Circuit קוונטי מופשט תמיד יכול להיות מיוצג על גרף זה, גם אם הקישוריות שלו מוגבלת, על ידי הצגת Gate SWAP להזזת המידע הקוונטי.

ה-Qubit מה-Circuits המופשטים שלנו נקראים _Qubit וירטואליים_ ואלה שבמפת הקישוריות הם _Qubit פיזיים_. ה-Transpiler מספק מיפוי בין Qubit וירטואליים לפיזיים. אחד הצעדים הראשונים בקומפילציה, שלב ה-_layout_, מבצע מיפוי זה.

> **Note:** למרות ששלב הניתוב שזור עם שלב ה-_layout_ — הבוחר את ה-Qubit בפועל — כברירת מחדל, נושא זה מתייחס אליהם כשלבים נפרדים לפשטות. השילוב של ניתוב ו-layout נקרא _מיפוי Qubit_.  למד עוד על שלבים אלה בנושא [שלבי Transpiler](transpiler-stages).

העבר את ארגומנט המילה המפתח `coupling_map` כדי לראות את השפעתו על ה-Transpiler:

In [4]:
coupling_map = target.build_coupling_map()

pass_manager = generate_preset_pass_manager(
    optimization_level=0, coupling_map=coupling_map, seed_transpiler=12345
)
qc_t_cm_lv0 = pass_manager.run(qc)
qc_t_cm_lv0.draw("mpl", idle_wires=False, fold=-1)

<Image src="../docs/images/guides/represent-quantum-computers/extracted-outputs/ec354bee-e06b-42ea-a117-6c1a9308ca73-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/represent-quantum-computers/extracted-outputs/ec354bee-e06b-42ea-a117-6c1a9308ca73-0.svg)

כפי שמוצג לעיל, הוכנסו מספר Gate SWAP (כל אחד מורכב משלושה Gate CX), מה שיגרום לשגיאות רבות במכשירים הנוכחיים. כדי לראות אילו Qubit נבחרים בטופולוגיית Qubit בפועל, השתמש ב-`plot_circuit_layout` מ-Qiskit Visualizations:

In [5]:
from qiskit.visualization import plot_circuit_layout

plot_circuit_layout(qc_t_cm_lv0, backend, view="physical")

<Image src="../docs/images/guides/represent-quantum-computers/extracted-outputs/9be74535-ed36-4d51-afeb-ee53c3f8a046-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/represent-quantum-computers/extracted-outputs/9be74535-ed36-4d51-afeb-ee53c3f8a046-0.svg)

זה מראה שה-Qubit הווירטואלי 0-11 שלנו מוּפּו בצורה טריוויאלית לשורת Qubit פיזיים 0-11. בואו נחזור לברירת המחדל (`optimization_level=1`), המשתמשת ב-`VF2Layout` אם נדרש ניתוב כלשהו.

In [6]:
pass_manager = generate_preset_pass_manager(
    optimization_level=1, coupling_map=coupling_map, seed_transpiler=12345
)
qc_t_cm_lv1 = pass_manager.run(qc)
qc_t_cm_lv1.draw("mpl", idle_wires=False, fold=-1)

<Image src="../docs/images/guides/represent-quantum-computers/extracted-outputs/8035fd05-f7cd-4151-b19a-4968202246e6-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/represent-quantum-computers/extracted-outputs/8035fd05-f7cd-4151-b19a-4968202246e6-0.svg)

כעת אין Gate SWAP שהוכנסו וה-Qubit הפיזיים שנבחרו זהים לאלה בעת השימוש במחלקת `target`.

In [7]:
from qiskit.visualization import plot_circuit_layout

plot_circuit_layout(qc_t_cm_lv1, backend, view="physical")

<Image src="../docs/images/guides/represent-quantum-computers/extracted-outputs/25d9fac3-abda-4b2d-81b4-351dc0772722-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/represent-quantum-computers/extracted-outputs/25d9fac3-abda-4b2d-81b4-351dc0772722-0.svg)

כעת ה-layout הוא בצורת טבעת. מכיוון ש-layout זה מכבד את הקישוריות של ה-Circuit, אין Gate SWAP, מה שמספק Circuit הרבה יותר טוב לביצוע.

## Gate בסיס

כל מחשב קוונטי תומך בסט הוראות מוגבל, הנקרא _Gate הבסיס_ שלו. כל Gate ב-Circuit חייב להיות מתורגם לאלמנטים של סט זה. סט זה צריך להיות מורכב מ-Gate חד- ודו-Qubit המספקים סט Gate אוניברסלי, כלומר כל פעולה קוונטית ניתנת לפירוק לאותם Gate. זה נעשה על ידי [BasisTranslator](../api/qiskit/qiskit.transpiler.passes.BasisTranslator), וניתן לציין את Gate הבסיס כארגומנט מילת מפתח ל-Transpiler לספק מידע זה.

In [8]:
basis_gates = list(target.operation_names)
print(basis_gates)

['sx', 'switch_case', 'x', 'if_else', 'measure', 'for_loop', 'delay', 'ecr', 'id', 'reset', 'rz']


The default single-qubit gates on _ibm_sherbrooke_ are `rz`, `x`, and `sx`, and the default two-qubit gate is `ecr` (echoed cross-resonance). CX gates are constructed from `ecr` gates, so on some QPUs `ecr` is specified as the two-qubit basis gate, while on others `cx` is the default. The `ecr` gate is the _entangling_ part of the `cx` gate. In addition to the control gates, there are also `delay` and `measurement` instructions.


<Admonition type="note">
    QPUs have default basis gates, but you can choose whatever gates you want, as long as you provide the instruction or add pulse gates (see [Create transpiler passes.](custom-transpiler-pass)) The default basis gates are those that calibrations have been done for on the QPU, so no further instruction/pulse gates need to be provided. For example, on some QPUs `cx` is the default two-qubit gate and `ecr` on others. See the list of possible [native gates and operations](/docs/guides/qpu-information#native-gates) for more details.
</Admonition>

In [9]:
pass_manager = generate_preset_pass_manager(
    optimization_level=1,
    coupling_map=coupling_map,
    basis_gates=basis_gates,
    seed_transpiler=12345,
)
qc_t_cm_bg = pass_manager.run(qc)
qc_t_cm_bg.draw("mpl", idle_wires=False, fold=-1)

<Image src="../docs/images/guides/represent-quantum-computers/extracted-outputs/313e4743-0.svg" alt="Output of the previous code cell" />

ה-Gate חד-Qubit ברירת המחדל על _ibm_sherbrooke_ הם `rz`, `x`, ו-`sx`, וה-Gate דו-Qubit ברירת המחדל הוא `ecr` (echoed cross-resonance). Gate CX בנויים מ-Gate `ecr`, לכן על חלק מה-QPU `ecr` מצוין כ-Gate הבסיס דו-Qubit, בעוד על אחרים `cx` הוא ברירת המחדל. Gate ה-`ecr` הוא החלק _המשזר_ של Gate ה-`cx`. בנוסף ל-Gate הבקרה, ישנן גם הוראות `delay` ומדידה.

> **Note:** ל-QPU יש Gate בסיס ברירת מחדל, אך תוכל לבחור כל Gate שתרצה, כל עוד אתה מספק את ההוראה או מוסיף Gate פולס (ראה [יצירת מעברי Transpiler.](custom-transpiler-pass)) Gate הבסיס ברירת המחדל הם אלה שכיול בוצע עליהם ב-QPU, לכן אין צורך לספק הוראות/Gate פולס נוספים. לדוגמה, על חלק מה-QPU `cx` הוא ה-Gate דו-Qubit ברירת המחדל ו-`ecr` על אחרים. ראה את רשימת ה-[Gate ופעולות ילידיות אפשריות](/guides/qpu-information#native-gates) לפרטים נוספים.

In [10]:
target["ecr"][(1, 0)]

InstructionProperties(duration=5.333333333333332e-07, error=0.007494257741828603)

![Output of the previous code cell](../docs/images/guides/represent-quantum-computers/extracted-outputs/313e4743-0.svg)

שים לב שאובייקטי `CXGate` פורקו ל-Gate `ecr` ו-Gate בסיס חד-Qubit.
## שיעורי שגיאת מכשיר
מחלקת `Target` יכולה להכיל מידע על שיעורי השגיאה לפעולות על המכשיר.
לדוגמה, הקוד הבא מאחזר את המאפיינים עבור Gate echoed cross-resonance (ECR) בין Qubit 1 ל-0 (שים לב שה-Gate ECR הוא כיווני):

In [11]:
from qiskit.transpiler import Target
from qiskit.circuit.controlflow import IfElseOp, SwitchCaseOp, ForLoopOp

err_targ = Target.from_configuration(
    basis_gates=basis_gates,
    coupling_map=coupling_map,
    num_qubits=target.num_qubits,
    custom_name_mapping={
        "if_else": IfElseOp,
        "switch_case": SwitchCaseOp,
        "for_loop": ForLoopOp,
    },
)

for i, (op, qargs) in enumerate(target.instructions):
    if op.name in basis_gates:
        err_targ[op.name][qargs] = target.instruction_properties(i)

Transpile with our new target `err_targ` as the target:

In [12]:
pass_manager = generate_preset_pass_manager(
    optimization_level=1, target=err_targ, seed_transpiler=12345
)
qc_t_cm_bg_et = pass_manager.run(qc)
qc_t_cm_bg_et.draw("mpl", idle_wires=False, fold=-1)

<Image src="../docs/images/guides/represent-quantum-computers/extracted-outputs/f1e270c4-e2cc-487e-a050-4180bc321b0b-0.svg" alt="Output of the previous code cell" />

הפלט מציג את משך ה-Gate (בשניות) ואת שיעור השגיאה שלו. כדי לחשוף מידע שגיאה ל-Transpiler, בנה מודל target עם `basis_gates` ו-`coupling_map` מלעיל ואכלס אותו בערכי שגיאה מ-Backend `FakeSherbrooke`.